In [ ]:
from defenders.pii_detection.crf.feature_builder import build_features
from defenders.pii_detection.crf.model import LinearCRF
import pandas as pd
from transformers import AutoTokenizer
from defenders.pii_detection.src.utils import prepare_dataset
from sklearn.metrics import classification_report,accuracy_score

In [ ]:
path_to_data = "./../data_pii/data.parquet"
manager = build_features()
df = pd.read_parquet(path_to_data)
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
df=prepare_dataset(df, tokenizer)
df["tokens"] = df["words"]
df["labels"] = df["labels"]
df = df[["tokens", "labels"]]

In [ ]:
df_non_o = df[df["labels"].apply(lambda labels: any(label != "O" for label in labels))]
df_all_o = df[df["labels"].apply(lambda labels: all(label == "O" for label in labels))]

In [ ]:
df = pd.concat([df_non_o.sample(n=7000, random_state=42),df_all_o.sample(n=1000, random_state=42)])
df=df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
dataset = df.values.tolist()
labels = set()
for _, y in dataset:
    labels.update(y)

labels = sorted(labels)

In [ ]:
model = LinearCRF(feature_manager=manager,labels=labels,lr=0.05,epochs=3,l2=1e-4)
model.fit(dataset)

In [ ]:
model.save_model("crf_from_Scratch_weights3.json")

In [ ]:
prediction = model.predict(["my","email","john@gmail.com"])
print(prediction)

In [ ]:
path_to_data = "./../data_pii/test.parquet"
manager = build_features()
df_test = pd.read_parquet(path_to_data)
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
df_test=prepare_dataset(df_test, tokenizer)
df_test["tokens"] = df_test["words"]
df_test["labels"] = df_test["labels"]
df_test = df_test[["tokens", "labels"]]

In [ ]:
def evaluate(model, df_test):
    all_predictions = []
    true_labels = []

    for i in range(len(df_test)):
        text = df_test["tokens"].iloc[i]
        predictions = model.predict(text)
        predicted_labels = [label[2:] for label in predictions]
        gold_labels = [label[2:] for label in df_test["labels"].iloc[i]]
        if len(predicted_labels) != len(gold_labels):
            continue

        all_predictions.extend(predicted_labels)
        true_labels.extend(gold_labels)

    accuracy = accuracy_score(true_labels, all_predictions)
    print(f"accuracy: {accuracy}")
    print(classification_report(true_labels, all_predictions))

In [ ]:
evaluate(model, df_test)